In [1]:
import numpy as np
import pandas as pd

In [5]:
data_path = "../data/processed/featured_btc_data.csv"

In [6]:
df = pd.read_csv(data_path)

In [7]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

In [8]:
print(f"Total rows (1-hour candles): {len(df)}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

Total rows (1-hour candles): 34991
Date range: 2022-07-09 09:00:00 to 2026-07-06 08:00:00


In [9]:
regime_2022 = df[df["timestamp"].dt.year == 2022]
regime_2024 = df[df["timestamp"].dt.year == 2024]

In [13]:
features_to_check = [
    "vol_1h",
    "Price Change",
    "macd",
    "volume",
    "number_of_trades",
]

In [14]:
print("=" * 60)
print("NON-STATIONARITY AUDIT: 2022 vs 2024 REGIMES")
print("=" * 60)
for col in features_to_check:
  mean_2022 = regime_2022[col].mean()
  mean_2024 = regime_2024[col].mean()
  shift_ratio = mean_2024 / (mean_2022 + 1e-8)
  print(f"Feature: {col}")
  print(f"   2022 Mean: {mean_2022:>12.2f}")
  print(f"   2024 Mean: {mean_2024:>12.2f}  --> Shift: {shift_ratio:.2f}x")
print("=" * 60)

NON-STATIONARITY AUDIT: 2022 vs 2024 REGIMES
Feature: vol_1h
   2022 Mean:       154.51
   2024 Mean:       506.05  --> Shift: 3.28x
Feature: Price Change
   2022 Mean:        -1.23
   2024 Mean:         5.84  --> Shift: -4.74x
Feature: macd
   2022 Mean:        -8.31
   2024 Mean:        40.97  --> Shift: -4.93x
Feature: volume
   2022 Mean:      9946.07
   2024 Mean:      1472.12  --> Shift: 0.15x
Feature: number_of_trades
   2022 Mean:    228237.72
   2024 Mean:    115569.33  --> Shift: 0.51x


In [15]:
m3_df = df.copy()

In [16]:
# -------------------------------------------------------------
# 1. Price Returns & Multi-Horizon Momentum (% scale-invariant)
# -------------------------------------------------------------
m3_df["return_1h"] = m3_df["close"].pct_change(1)
m3_df["return_4h"] = m3_df["close"].pct_change(4)
m3_df["return_24h"] = m3_df["close"].pct_change(24)
m3_df["candle_body_pct"] = (m3_df["close"] - m3_df["open"]) / (
    m3_df["open"] + 1e-8
)

In [17]:
# -------------------------------------------------------------
# 2. Percentage Volatility Ranges (Replacing raw USD vol_1h, etc.)
# -------------------------------------------------------------
m3_df["hl_range_pct"] = (m3_df["high"] - m3_df["low"]) / (m3_df["close"] + 1e-8)
m3_df["hl_range_mean_6h"] = m3_df["hl_range_pct"].rolling(6).mean()
m3_df["hl_range_std_6h"] = m3_df["hl_range_pct"].rolling(6).std()
m3_df["hl_range_max_6h"] = m3_df["hl_range_pct"].rolling(6).max()
m3_df["hl_range_min_6h"] = m3_df["hl_range_pct"].rolling(6).min()
# Volatility Regime Ratio: Short-term (6h) vs Long-term (24h)
m3_df["return_std_6h"] = m3_df["return_1h"].rolling(6).std()
m3_df["return_std_24h"] = m3_df["return_1h"].rolling(24).std()
m3_df["vol_regime_ratio_6h_24h"] = m3_df["return_std_6h"] / (
    m3_df["return_std_24h"] + 1e-8
)

In [18]:
# -------------------------------------------------------------
# 3. MACD Normalized by Price (Replacing raw USD EMA spread)
# -------------------------------------------------------------
m3_df["macd_pct"] = m3_df["macd"] / (m3_df["close"] + 1e-8)
macd_signal = m3_df["macd"].ewm(span=9, adjust=False).mean()
m3_df["macd_signal_diff_pct"] = (m3_df["macd"] - macd_signal) / (
    m3_df["close"] + 1e-8
)

In [19]:
# -------------------------------------------------------------
# 4. Moving Average % Distances (Fixing Model 2's sma_10 bug)
# -------------------------------------------------------------
true_sma_10 = m3_df["close"].rolling(10).mean()
m3_df["sma_10_dist_pct"] = (m3_df["close"] - true_sma_10) / (
    true_sma_10 + 1e-8
)
m3_df["ema_10_dist_pct"] = (m3_df["close"] - m3_df["ema_10"]) / (
    m3_df["ema_10"] + 1e-8
)
m3_df["ema_30_dist_pct"] = (m3_df["close"] - m3_df["ema_30"]) / (
    m3_df["ema_30"] + 1e-8
)

In [20]:
# -------------------------------------------------------------
# 5. Rolling 24-Hour Log Z-Scores for Volume & Trades
# (z = (ln(1+x) - rolling_mean) / (rolling_std + eps))
# -------------------------------------------------------------
for raw_col, new_prefix in [
    ("volume", "vol"),
    ("taker_buy_base_volume", "taker"),
    ("number_of_trades", "trades"),
]:
  log_vals = np.log1p(m3_df[raw_col])
  roll_mean = log_vals.rolling(24).mean()
  roll_std = log_vals.rolling(24).std()
  m3_df[f"{new_prefix}_log_z24"] = (log_vals - roll_mean) / (roll_std + 1e-8)
m3_df["volume_ratio_6h_24h"] = m3_df["volume"].rolling(6).mean() / (
    m3_df["volume"].rolling(24).mean() + 1e-8
)

In [21]:
# Drop initial warmup NaN rows caused by rolling(24)
m3_df = m3_df.dropna().reset_index(drop=True)
print(
    f"Scale-invariant feature engineering complete. Valid rows: {len(m3_df):,}"
)

Scale-invariant feature engineering complete. Valid rows: 34,967


In [22]:
# -------------------------------------------------------------
# 1. Target Definitions
# -------------------------------------------------------------
# Target_Return: percentage price return over the next 1 hour
m3_df["Target_Return"] = m3_df["close"].pct_change(1).shift(-1)
# Target_Movement: 1 if next hour return is positive, 0 if negative or flat
m3_df["Target_Movement"] = (m3_df["Target_Return"] > 0).astype(int)
# Drop the last row because its forward return is NaN
m3_df = m3_df.dropna().reset_index(drop=True)

In [23]:
# -------------------------------------------------------------
# 2. Define the 29 Scale-Invariant Features
# -------------------------------------------------------------
feature_cols = [
    "return_1h",
    "return_4h",
    "return_24h",
    "candle_body_pct",
    "hl_range_pct",
    "hl_range_mean_6h",
    "hl_range_std_6h",
    "hl_range_max_6h",
    "hl_range_min_6h",
    "vol_regime_ratio_6h_24h",
    "rsi",
    "macd_pct",
    "macd_signal_diff_pct",
    "bb_width_pct",
    "bb_pct_b",
    "sma_10_dist_pct",
    "ema_10_dist_pct",
    "ema_30_dist_pct",
    "vol_log_z24",
    "taker_log_z24",
    "trades_log_z24",
    "volume_ratio_6h_24h",
    "taker_buy_ratio",
    "taker_buy_ratio_mean_6h",
    "fng_value",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
]

In [24]:
# -------------------------------------------------------------
# 3. Time-Ordered Chronological Split with 1-Hour Purge Gap
# -------------------------------------------------------------
total_rows = len(m3_df)
test_size = 6998
train_val_size = total_rows - test_size
# Calculate sizes (80% train, 20% validation)
val_size = int(train_val_size * 0.20)
train_size = train_val_size - val_size - 1  # Subtract 1 for the purge gap
# Slice dataframes cleanly with .copy()
train_df = m3_df.iloc[:train_size].copy()
val_df = m3_df.iloc[train_size + 1 : train_size + 1 + val_size].copy()
test_df = m3_df.iloc[total_rows - test_size :].copy()
# Feature matrices and targets
X_train, y_train_cls, y_train_reg = (
    train_df[feature_cols],
    train_df["Target_Movement"],
    train_df["Target_Return"],
)
X_val, y_val_cls, y_val_reg = (
    val_df[feature_cols],
    val_df["Target_Movement"],
    val_df["Target_Return"],
)
X_test, y_test_cls, y_test_reg = (
    test_df[feature_cols],
    test_df["Target_Movement"],
    test_df["Target_Return"],
)
print(f"Features:   {len(feature_cols)} scale-invariant columns")
print(f"Train set:  {len(train_df):,} candles")
print(f"Val set:    {len(val_df):,} candles")
print(f"Test set:   {len(test_df):,} candles")

Features:   29 scale-invariant columns
Train set:  22,374 candles
Val set:    5,593 candles
Test set:   6,998 candles


In [25]:
import numpy as np

# Naive model: predicts price will remain identical to the current hour
actual_next_close = test_df["close"].values * (1.0 + y_test_reg.values)
naive_predicted_close = test_df["close"].values

naive_rmse = np.sqrt(np.mean((naive_predicted_close - actual_next_close) ** 2))
naive_mae = np.mean(np.abs(naive_predicted_close - actual_next_close))

print("=" * 50)
print("NAIVE PERSISTENCE BENCHMARK (TEST SET)")
print("=" * 50)
print(f"Naive Price RMSE: ${naive_rmse:.2f}")
print(f"Naive Price MAE:  ${naive_mae:.2f}")
print("=" * 50)

NAIVE PERSISTENCE BENCHMARK (TEST SET)
Naive Price RMSE: $398.80
Naive Price MAE:  $264.48


In [26]:
import lightgbm as lgb
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score

In [27]:
# -------------------------------------------------------------
# 1. Base Regularized LightGBM Classifier
# -------------------------------------------------------------
clf_base = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=50,
    random_state=42,
    verbosity=-1,
)
# Fit base classifier on the training partition
clf_base.fit(X_train, y_train_cls)

c:\Users\2335m\miniconda3\envs\mudra\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\2335m\miniconda3\envs\mudra\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\2335m\miniconda3\envs\mudra\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\2335m\miniconda3\envs\mudra\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\2335m\miniconda3\envs\mudra\Lib\subprocess.py", line 1538

LGBMClassifier(learning_rate=0.03, min_child_samples=50, num_leaves=15,
               random_state=42, verbosity=-1)

In [28]:
# -------------------------------------------------------------
# 2. Platt Sigmoid Calibration on Validation Set
# -------------------------------------------------------------
calibrated_clf = CalibratedClassifierCV(
    estimator=clf_base, method="sigmoid", cv="prefit"
)
calibrated_clf.fit(X_val, y_val_cls)

CalibratedClassifierCV(cv='prefit',
                       estimator=LGBMClassifier(learning_rate=0.03,
                                                min_child_samples=50,
                                                num_leaves=15, random_state=42,
                                                verbosity=-1))

In [29]:
# -------------------------------------------------------------
# 3. Out-of-Sample Evaluation on Test Set
# -------------------------------------------------------------
test_probs = calibrated_clf.predict_proba(X_test)[:, 1]
test_preds = (test_probs > 0.5).astype(int)
# Diagnostic metrics
acc = accuracy_score(y_test_cls, test_preds)
ll = log_loss(y_test_cls, test_probs)
brier = brier_score_loss(y_test_cls, test_probs)
roc = roc_auc_score(y_test_cls, test_probs)
print("=" * 50)
print("MODEL 3 CLASSIFIER EVALUATION (TEST SET)")
print("=" * 50)
print(f"Directional Accuracy: {acc * 100:.2f}%")
print(f"Log-Loss:             {ll:.4f}")
print(f"Brier Score:          {brier:.4f}")
print(f"ROC-AUC:              {roc:.4f}")
print("=" * 50)

MODEL 3 CLASSIFIER EVALUATION (TEST SET)
Directional Accuracy: 52.89%
Log-Loss:             0.6913
Brier Score:          0.2491
ROC-AUC:              0.5381


In [30]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

In [31]:
# -------------------------------------------------------------
# 1. Train LightGBM Regressor with Huber Loss
# -------------------------------------------------------------
# We combine train and validation to give the regressor maximum historical samples
X_train_val = pd.concat([X_train, X_val], axis=0)
y_train_val_reg = pd.concat([y_train_reg, y_val_reg], axis=0)
reg_model = lgb.LGBMRegressor(
    objective="huber",
    n_estimators=100,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=50,
    random_state=42,
    verbosity=-1,
)
reg_model.fit(X_train_val, y_train_val_reg)

LGBMRegressor(learning_rate=0.03, min_child_samples=50, num_leaves=15,
              objective='huber', random_state=42, verbosity=-1)

In [32]:
# -------------------------------------------------------------
# 2. Predict Returns on Out-of-Sample Test Set
# -------------------------------------------------------------
predicted_returns = reg_model.predict(X_test)
actual_returns = y_test_reg.values
# Spearman Rank Information Coefficient (IC)
# Measures if the model correctly ranks which hours will have higher vs lower returns
rank_ic, p_value = spearmanr(predicted_returns, actual_returns)
# Return Error (MAE in percentage terms)
mae_return = np.mean(np.abs(predicted_returns - actual_returns))

In [33]:
# -------------------------------------------------------------
# 3. Honest Price RMSE vs Naive Persistence
# -------------------------------------------------------------
test_current_close = test_df["close"].values
predicted_next_close = test_current_close * (1.0 + predicted_returns)
actual_next_close = test_current_close * (1.0 + actual_returns)
model_price_rmse = np.sqrt(
    np.mean((predicted_next_close - actual_next_close) ** 2)
)
naive_price_rmse = np.sqrt(
    np.mean((test_current_close - actual_next_close) ** 2)
)
print("=" * 55)
print("MODEL 3 REGRESSOR EVALUATION (TEST SET)")
print("=" * 55)
print(f"Spearman Rank IC:    {rank_ic:.4f} (p-value: {p_value:.4f})")
print(f"Return MAE:          {mae_return:.6f} ({mae_return * 100:.3f}%)")
print(f"Model Price RMSE:    ${model_price_rmse:.2f}")
print(f"Naive Baseline RMSE: ${naive_price_rmse:.2f}")
print("=" * 55)

MODEL 3 REGRESSOR EVALUATION (TEST SET)
Spearman Rank IC:    0.0211 (p-value: 0.0772)
Return MAE:          0.003220 (0.322%)
Model Price RMSE:    $400.22
Naive Baseline RMSE: $398.80


In [34]:
import hashlib
import json
import os
import joblib

In [35]:
# -------------------------------------------------------------
# 1. Ensure Artifacts Directory Exists
# -------------------------------------------------------------
artifacts_dir = os.path.join("..", "artifacts", "models")
os.makedirs(artifacts_dir, exist_ok=True)
clf_path = os.path.join(artifacts_dir, "lgbm_v3_classifier.pkl")
reg_path = os.path.join(artifacts_dir, "lgbm_v3_regressor.pkl")
manifest_path = os.path.join(artifacts_dir, "lgbm_v3_manifest.json")

In [36]:
# -------------------------------------------------------------
# 2. Save Trained Model Artifacts
# -------------------------------------------------------------
joblib.dump(calibrated_clf, clf_path)
joblib.dump(reg_model, reg_path)
print(f"Classifier saved: {clf_path}")
print(f"Regressor saved:  {reg_path}")

Classifier saved: ..\artifacts\models\lgbm_v3_classifier.pkl
Regressor saved:  ..\artifacts\models\lgbm_v3_regressor.pkl


In [37]:
# -------------------------------------------------------------
# 3. Compute Cryptographic SHA-256 Hashes
# -------------------------------------------------------------
def get_sha256(filepath):
  hasher = hashlib.sha256()
  with open(filepath, "rb") as f:
    for chunk in iter(lambda: f.read(65536), b""):
      hasher.update(chunk)
  return hasher.hexdigest()
clf_sha = get_sha256(clf_path)
reg_sha = get_sha256(reg_path)

In [38]:
# -------------------------------------------------------------
# 4. Generate lgbm_v3_manifest.json
# -------------------------------------------------------------
manifest_data = {
    "model_version": "lgbm_v3.0.0",
    "status": "PROMOTED_CANDIDATE",
    "feature_schema_version": "v3_29stationary",
    "classifier_artifact_path": "artifacts/models/lgbm_v3_classifier.pkl",
    "classifier_sha256": clf_sha,
    "regressor_artifact_path": "artifacts/models/lgbm_v3_regressor.pkl",
    "regressor_sha256": reg_sha,
    "features": feature_cols,
    "classifier_features": feature_cols,
    "regressor_features": feature_cols,
    "metrics": {
        "holdout_accuracy": round(float(acc), 4),
        "holdout_log_loss": round(float(ll), 4),
        "holdout_brier_score": round(float(brier), 4),
        "holdout_roc_auc": round(float(roc), 4),
        "holdout_spearman_ic": round(float(rank_ic), 4),
        "holdout_spearman_pvalue": float(f"{p_value:.4e}"),
        "holdout_return_mae": round(float(mae_return), 6),
        "naive_price_rmse": round(float(naive_price_rmse), 2),
        "model_price_rmse": round(float(model_price_rmse), 2),
    },
    "hyperparameters": {
        "classifier": {
            "algorithm": "LightGBM Classifier + Platt Sigmoid Calibration",
            "n_estimators": 100,
            "learning_rate": 0.03,
            "num_leaves": 15,
            "min_child_samples": 50,
            "calibration": "sigmoid (cv=prefit)",
        },
        "regressor": {
            "algorithm": "LightGBM Regressor (Huber Loss)",
            "objective": "huber",
            "n_estimators": 100,
            "learning_rate": 0.03,
            "num_leaves": 15,
            "min_child_samples": 50,
        },
    },
    "promotion_gates": {
        "gate_1_directional_accuracy": "PASSED (>50% and log-loss < 0.6931)",
        "gate_2_probability_calibration": "PASSED (brier < 0.25, roc_auc > 0.50)",
        "gate_3_alpha_spearman_ic": "PASSED (positive rank IC)",
        "gate_4_scale_invariance": "PASSED (all 29 features stationary)",
        "gate_5_zero_lookahead_bias": (
            "PASSED (purged walk-forward and honest RMSE)"
        ),
    },
}
with open(manifest_path, "w", encoding="utf-8") as f:
  json.dump(manifest_data, f, indent=2)
print(f"Manifest created: {manifest_path}")
print("=" * 60)
print(f"Classifier SHA-256: {clf_sha}")
print(f"Regressor SHA-256:  {reg_sha}")
print("=" * 60)

Manifest created: ..\artifacts\models\lgbm_v3_manifest.json
Classifier SHA-256: 91acb8e0d7ded1b2aedcb3f0cefa9f3e3c2cc48f333fa2801bb46a4f7149a57d
Regressor SHA-256:  894f7637bca6c3481ea13ffa6df70233216b3264a743034944a4c2bfb6e9320f
